In [ ]:
import itertools
import json
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.datasets import dataset_ref
from uavloc.filtering.ins_error import InsErrorPrior
from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.arms import FilterParams, build_correlator, estimator_factory
from uavloc.simulation.sequence import (all_fixes, flight_metrics, load_flight_set, per_flight_table, run_flights,
                                        village_bootstrap)
from uavloc.track import REPO_ROOT, experiment_dir, load_runs, log_run, set_seed

# Params

In [ ]:
EXP = "028-ins-error-state"
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
INITIAL_ERROR_M = {"normal": 0.0, "cold": 40.0}
SIGMA0_M = {"normal": 1.0, "cold": 25.0}
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
N_PARTICLES = 2000
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
TUNING_DIR = "artifacts/tuning"
SCENARIO = "normal"
SPLIT = "test"
LOG_RUN = True
WAIT_FOR_RUNS = 0
REGISTERED_RUNS = {"normal": "028.1", "cold": "028.2"}
BASELINE_RUNS = {"normal": "027.4", "cold": "027.9"}
BASELINE_EXP = "027-sequence-filters"
TUNED = {"temperature": 0.5, "q_pos_m_per_sqrt_s": 0.1}   # from tuning, set per papermill run
BOOTSTRAP_RESAMPLES = 10_000
NOTE = ""

In [ ]:
set_seed(SEED)

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
tuning = pd.concat([pd.read_csv(p) for p in sorted(Path(TUNING_DIR).glob("shard-[0-9].csv"))], ignore_index=True)
baseline = pd.read_csv(experiment_dir(BASELINE_EXP) / "artifacts" / f"flights-{BASELINE_RUNS[SCENARIO]}.csv")
len(specs), len(tuning), len(baseline)

# Model

In [ ]:
def filter_params(temperature: float, q_pos: float) -> FilterParams:
    return FilterParams(temperature, q_pos, N_PARTICLES, 0, 0, 0.0, 0.0, 0.0, 0.0, WINDOW_CAP_HALF_M, WINDOW_MARGIN_M)

In [ ]:
def correlator_for(map_raster):
    return build_correlator(map_raster, CORRELATOR, DEVICE)

In [ ]:
def ins_prior(ins) -> InsErrorPrior:
    return InsErrorPrior(ins.velocity_bias_mps, ins.velocity_random_walk_mps, ins.heading_bias_deg, ins.speed_scale_std)

In [ ]:
params = filter_params(TUNED["temperature"], TUNED["q_pos_m_per_sqrt_s"])
factories = {"ins-error": estimator_factory("ins-error", params, ins_prior=ins_prior(set_config.ins))}

# Run

In [ ]:
outcomes = run_flights(set_config, specs, INITIAL_ERROR_M[SCENARIO], SIGMA0_M[SCENARIO], correlator_for, factories)

In [ ]:
table = per_flight_table(outcomes["ins-error"])
metrics = flight_metrics(outcomes["ins-error"])
pd.Series(metrics)

In [ ]:
comparison = {}
for column in ("end_error_m", "mean_error_m", "diverged"):
    delta, low, high = village_bootstrap(table, baseline, column, BOOTSTRAP_RESAMPLES, SEED)
    comparison.update({f"vs_baseline_{column}_delta": delta, f"vs_baseline_{column}_ci_low": low,
                       f"vs_baseline_{column}_ci_high": high})
comparison["baseline_mean_end_error_m"] = float(baseline["end_error_m"].mean())
comparison["ratio_to_baseline_mean_end_error"] = metrics["mean_end_error_m"] / comparison["baseline_mean_end_error_m"]
comparison

# Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_end_error_cdf({f"027 particles ({BASELINE_RUNS[SCENARIO]})": baseline["end_error_m"], "ins-error particles": table["end_error_m"],
                    "INS only": table["end_error_ins_m"]}, ax)
ax.set_title(f"028, {SCENARIO} start, {SPLIT} flights ({len(specs)})")
fig.tight_layout()

# Log

In [ ]:
config = {"registered_run": REGISTERED_RUNS[SCENARIO], "arm": "ins-error", "scenario": SCENARIO, "split": SPLIT,
          "route_set": dataset_ref(ROUTE_SET), "initial_error_m": INITIAL_ERROR_M[SCENARIO], "sigma0_m": SIGMA0_M[SCENARIO],
          "ins": asdict(set_config.ins), "ins_prior": asdict(ins_prior(set_config.ins)), "correlator": CORRELATOR,
          "filter": asdict(params), "baseline_run": f"{BASELINE_EXP}/{BASELINE_RUNS[SCENARIO]}",
          "tuning": {"grid": {"temperature": sorted(tuning["temperature"].unique().tolist()),
                              "q_pos_m_per_sqrt_s": sorted(tuning["q_pos_m_per_sqrt_s"].unique().tolist())},
                     "chosen": TUNED, "tuning_mean_end_error_m": float(tuning["mean_end_error_m"].min())}}

In [ ]:
if LOG_RUN:
    table.to_csv(Path("artifacts") / f"flights-{REGISTERED_RUNS[SCENARIO]}.csv", index=False)
    all_fixes(outcomes["ins-error"]).to_csv(Path("artifacts") / f"fixes-{REGISTERED_RUNS[SCENARIO]}.csv", index=False)
    while len(load_runs(EXP)) < WAIT_FOR_RUNS:
        time.sleep(30)
    expected = f"028.{len(load_runs(EXP)) + 1}"
    if expected != REGISTERED_RUNS[SCENARIO]:
        raise RuntimeError(f"run would log as {expected}, registered as {REGISTERED_RUNS[SCENARIO]}")
    log_run(exp=EXP, config=config, metrics={**metrics, **comparison},
            note=f"{REGISTERED_RUNS[SCENARIO]}: ins-error particles, {SCENARIO} start. {NOTE}", figures=[fig],
            seed=SEED, device=DEVICE)